# Распаковка составного проектора

In [10]:
import json
import re
import ast

with open("out1.json", encoding="utf-8") as f:
    data = json.load(f)

pattern = re.compile(r"^(\(.+?\)) — (\[.+\])$")

result_proj = {}
for line in data["wp_bpe_comparison_str"]:
    m = pattern.match(line)
    key_str, val_str = m.group(1), m.group(2)
    key = ast.literal_eval(key_str)   # tuple (token, id)
    value = ast.literal_eval(val_str) # list[tuple]
    result_proj[key] = value

# Распаковка MLP-проектора

In [15]:
import json
import re
import ast

with open("vocab_mapping_mlp.json", encoding="utf-8") as f:
    data = json.load(f)

pattern = re.compile(r"^(\(.+?\)) — (\[.+\])$")

result_mlp = {}
for line in data:
    m = pattern.match(line)
    key_str, val_str = m.group(1), m.group(2)
    key = ast.literal_eval(key_str)
    value = ast.literal_eval(val_str)
    result_mlp[key] = value

# Установка моделей

In [16]:
from enum import Enum
from tokenizers import Tokenizer
from torch.utils.data.dataset import Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM
from tokenizers.models import BPE, Unigram
from torch.utils.data import DataLoader
import torch

/home/misha/Desktop/knowledge_distillation/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [17]:
model_name_1 = "LiquidAI/LFM2.5-230M"
tokenizer_1 = Tokenizer.from_pretrained(model_name_1)
print(f'Длина словаря {model_name_1}: {tokenizer_1.get_vocab_size()}')
print(f'Модель токенизатора {model_name_1}: {tokenizer_1.model}')

Длина словаря LiquidAI/LFM2.5-230M: 64402
Модель токенизатора LiquidAI/LFM2.5-230M: BPE(dropout=None, unk_token=None, continuing_subword_prefix=None, end_of_word_suffix=None, fuse_unk=False, byte_fallback=False, ignore_merges=False, vocab={"<|pad|>":0, "<|startoftext|>":1, "<|endoftext|>":2, "<|fim_pre|>":3, "<|fim_mid|>":4, ...}, merges=[("Ċ", "Ċ"), ("Ċ", "ĊĊ"), ("ĊĊ", "Ċ"), ("Ċ", "ĊĊĊ"), ("ĊĊ", "ĊĊ"), ...])


In [18]:
model_name_2 = "Qwen/Qwen2.5-0.5B-Instruct"
tokenizer_2 = Tokenizer.from_pretrained(model_name_2)
print(f'Длина словаря {model_name_2}: {tokenizer_2.get_vocab_size()}')
print(f'Модель токенизатора {model_name_2}: {tokenizer_2.model}')

Длина словаря Qwen/Qwen2.5-0.5B-Instruct: 151665
Модель токенизатора Qwen/Qwen2.5-0.5B-Instruct: BPE(dropout=None, unk_token=None, continuing_subword_prefix="", end_of_word_suffix="", fuse_unk=False, byte_fallback=False, ignore_merges=False, vocab={"!":0, """:1, "#":2, "$":3, "%":4, ...}, merges=[("Ġ", "Ġ"), ("ĠĠ", "ĠĠ"), ("i", "n"), ("Ġ", "t"), ("ĠĠĠĠ", "ĠĠĠĠ"), ...])


# Установка токенизаторов

In [21]:
tokenizer_model_1 = AutoTokenizer.from_pretrained(model_name_1)
model_1 = AutoModelForCausalLM.from_pretrained(model_name_1,
                                               torch_dtype=torch.float16,
                                               device_map="auto")

Loading weights: 100%|██████████| 132/132 [00:00<00:00, 2772.45it/s]


In [22]:
tokenizer_model_2 = AutoTokenizer.from_pretrained(model_name_2)
model_2 = AutoModelForCausalLM.from_pretrained(model_name_2,
                                               torch_dtype=torch.float16,
                                               device_map="auto")

Loading weights: 100%|██████████| 290/290 [00:00<00:00, 3018.21it/s]


In [ ]:
class Benchmark:
    def __init__(self, )